In [ ]:
# Importing necesaary libraries
import cv2
import random
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Path to the folder containing the dataset
input_folder = r"C:\Users\Priyanka\Desktop\SEM5\IP\dataset"
output_folder = r"C:\Users\Priyanka\Desktop\SEM5\IP\dataset_out"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

In [ ]:
K = 3  # Number of clusters
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1.0)
attempts = 10
kernel = np.ones((5, 5), np.uint8)

# Function to segment one image with K-means and return segmented image
def segment_image(img):

    # Convert the input image from BGR color space to LAB color space
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)

    # Reshape to a 2D array where each pixel is a row with 3 color components
    img_flat = img_lab.reshape((-1, 3)).astype(np.float32)

    # Apply k-means clustering to cluster pixel colors into K clusters
    _, labels, centers = cv2.kmeans(img_flat, K, None, criteria, attempts, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)

    # Assign each data point to the nearest centroid based on a distance metric
    segmented_img_flat = centers[labels.flatten()]
    segmented_img = segmented_img_flat.reshape(img.shape)
    segmented_img_rgb = cv2.cvtColor(segmented_img, cv2.COLOR_Lab2RGB)

    return segmented_img_rgb

In [ ]:
# Function to perform morphological operations
def morph_postprocess(segmented_img, kernel_size=9, erosion_iters=1, dilation_iters=1):
    # Convert segment centers to grayscale labels for morphological processing
    segmented_gray = cv2.cvtColor(segmented_img, cv2.COLOR_RGB2GRAY)
    
    # Threshold to create a binary image. Adjust threshold according to your image
    _, binary_img = cv2.threshold(segmented_gray, 1, 255, cv2.THRESH_BINARY)
    
    # Define a kernel (structuring element) for morphology
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    
    # Apply morphological opening (erosion followed by dilation) to remove noise
    #opening = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)
    
    # Apply morphological closing (dilation followed by erosion) to close gaps
    #closing = cv2.morphologyEx(binary_img, cv2.MORPH_CLOSE, kernel)

    # Apply erosion to shrink object boundaries and remove noise
    #eroded = cv2.erode(binary_img, kernel, iterations=erosion_iters)
    
    # Apply dilation to grow object boundaries and fill gaps
    dilated = cv2.dilate(binary_img, kernel, iterations=dilation_iters)
    
    # Use the processed mask to refine the original segmented image
    refined = cv2.bitwise_and(segmented_img, segmented_img, mask=dilated)
    
    return refined

In [ ]:
# Loop through all images in folder
image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Randomly pick 6 images (no repetition)
random_images = random.sample(image_files, 6)

for image_file in random_images:
    image_file = image_file.strip().rstrip('.')  # Remove trailing spaces and dots
    img_path = os.path.join(input_folder, image_file)
    #img = cv2.imread(img_path)
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)  # Read as is, including grayscale

    # To check if the file exists or not
    if not os.path.exists(img_path):
        print(f"File does not exist: {img_path}")
        continue

    # To know when the image can't be read
    if img is None:
        print(f"Warning: Could not read image {img_path}. Skipping.")
        continue

    # If the image is grayscale or has a single channel, convert to 3-channel BGR
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    segmented_img = segment_image(img)
    refined = morph_postprocess(segmented_img, kernel_size=10)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Plot original, segmented and refined images side by side
    plt.figure(figsize=(9,3))
    plt.suptitle(image_file)
    
    plt.subplot(1, 3, 1)
    plt.imshow(img_rgb)
    plt.title('Original image')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(segmented_img)
    plt.title('Segmented image')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(refined)
    plt.title('Refined image')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    plt.close()